In [1]:
!pip install ultralytics supervision opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.4/212.4 kB 15.3 MB/s eta 0:00:00


In [2]:
import cv2
import random
from ultralytics import YOLO
import supervision as sv
from google.colab.patches import cv2_imshow
from IPython.display import HTML
from base64 import b64encode

# Fixed color generator (no flicker)
def get_color(cls):
    COLORS = [
        (255, 0, 0),    # person
        (0, 255, 0),    # bicycle
        (0, 0, 255),    # car
        (255, 255, 0),  # motorcycle
        (255, 0, 255),  # bus
        (0, 255, 255),  # truck
    ]
    return COLORS[cls % len(COLORS)]



Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# Medium model = best for football + people
model = YOLO("yolov8m.pt")


In [4]:
VIDEO_PATH = "./sample.mp4"   # <-- your input video
OUTPUT_PATH = "output_detected.mp4"


In [5]:
cap = cv2.VideoCapture(VIDEO_PATH)
video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)


In [6]:
with sv.VideoSink(OUTPUT_PATH, video_info) as sink:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # YOLOv8 Multi-class Tracking
        results = model.track(
            frame,
            persist=True,
            imgsz=640,
            conf=0.3,
            classes=[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,
                     21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,
                     39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,
                     57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,
                     75,76,77,78,79,80],  # person + vehicles
            tracker="bytetrack.yaml"
        )

        annotated = frame.copy()

        if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu()
            ids = results[0].boxes.id.int().cpu().tolist()
            classes = results[0].boxes.cls.int().cpu().tolist()

            for box, track_id, cls in zip(boxes, ids, classes):
                x1, y1, x2, y2 = map(int, box)
                label = model.names[cls]
                color = get_color(cls)

                cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
                cv2.putText(
                    annotated,
                    f"{label} | ID {track_id}",
                    (x1, max(y1 - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    color,
                    2
                )

        sink.write_frame(annotated)

cap.release()
print("✅ Detection finished. Video saved.")


requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 232ms
Prepared 1 package in 87ms
Installed 1 package in 3ms
 + lap==0.5.12

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


0: 384x640 12 cars, 1255.6ms
Speed: 3.2ms preprocess, 1255.6ms inference, 48.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 11 cars, 924.2ms
Speed: 3.0ms preprocess, 924.2ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 cars, 963.7ms
Speed: 2.2ms preprocess, 963.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 929.7ms
Speed: 2.1ms preprocess, 929.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 13 cars, 904.7ms
Speed: 2.1ms preprocess, 904.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 3